# Fuzzy C-Means - Starter Notebook
This notebook implements fuzzy C-means from scratch so each point can belong to multiple clusters with different membership strengths.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

In [ ]:
X, _ = make_blobs(n_samples=240, centers=3, cluster_std=1.1, random_state=42)
n_clusters = 3
m = 2.0
max_iter = 100
tolerance = 1e-4
rng = np.random.default_rng(42)

membership = rng.random((len(X), n_clusters))
membership = membership / membership.sum(axis=1, keepdims=True)

In [ ]:
for _ in range(max_iter):
    membership_power = membership ** m
    centers = (membership_power.T @ X) / membership_power.sum(axis=0)[:, None]

    distances = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2) + 1e-8
    new_membership = np.zeros_like(membership)
    for cluster_index in range(n_clusters):
        ratio = (distances[:, cluster_index][:, None] / distances) ** (2 / (m - 1))
        new_membership[:, cluster_index] = 1 / ratio.sum(axis=1)

    if np.linalg.norm(new_membership - membership) < tolerance:
        membership = new_membership
        break
    membership = new_membership

hard_labels = membership.argmax(axis=1)
membership[:5]

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=hard_labels, cmap='viridis', s=35, alpha=0.8)
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=220)
plt.title('Fuzzy C-means Clustering')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

print('Sample membership degrees for first five points:')
print(np.round(membership[:5], 3))